# 08 — Multitemporal LULC-segmentering med Prithvi-EO

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_090-Multitemporal-LULC.ipynb)

**Syfte:** Demonstrera SDL 3.0:s flaggskeppspipeline — 23-klassers markanvändning/marktäcke (LULC) via multitemporal Prithvi-EO 2.0 foundation model. Denna notebook visar hur Naturvårdsverkets NMD kombineras med Jordbruksverkets LPIS och Skogsstyrelsens avverkningsanmälan i ett enhetligt schema.

**Partners som bidragit:**
- **Naturvårdsverket** — NMD (Nationella Marktäckedata)
- **Jordbruksverket** — LPIS (Land Parcel Identification System) — 12 grödor
- **Skogsstyrelsen** — avverkningsanmälan (SKS)
- **RISE** — Prithvi-fine-tuning, multitemporal träningspipeline
- **IBM/NASA** — Prithvi-EO 2.0 foundation model (Apache 2.0)

**Datakällor:**
- Copernicus Sentinel-2 L2A — 4 temporala ramar per tile
- NMD 2018 + LPIS 2018–2024 + SKS 2021–2026
- Prithvi-EO 2.0 backbone (300M params)

**Licens:** CC0 1.0 (notebook) · Prithvi Apache 2.0

## 1. 23-klassers Unified Schema

| Klass | Namn | Källa |
|-------|------|-------|
| 0 | bakgrund | — |
| 1–5 | tallskog, granskog, lövskog, blandskog, sumpskog | NMD |
| 6 | tillfälligt ej skog | NMD |
| 7 | våtmark | NMD |
| 8 | öppen mark | NMD |
| 9 | bebyggelse | NMD |
| 10 | vatten | NMD |
| 11–21 | vete, korn, havre, oljeväxter, slåttervall, bete, potatis, sockerbetor, trindsäd, råg, majs | LPIS |
| 22 | hygge (avverkning) | SKS |

**Multitemporal struktur — 4 ramar per tile:**
1. **Höst (Sep–Okt från år-1)** — stubble, höstsäd
2. **Tidig växtsäsong (VPP-styrd)** — tillväxtstart per latitud
3. **Peak vegetation** — max NDVI
4. **Sen växtsäsong / skörd** — skillnader mellan grödor

Varje ram har 6 spektrala band (B02, B03, B04, B8A, B11, B12) + 11 hjälpkanaler (trädata, DEM, VPP-fenologi, avverkningssannolikhet). Total input: **35 kanaler**.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Skånes mosaiklandskap — blandning av åker, skog, bebyggelse
AOI = {
    "west": 13.20,
    "south": 55.80,
    "east": 13.40,
    "north": 55.95,
}
YEAR = 2024
REFERENCE_DATE = f"{YEAR}-07-15"  # Peak vegetation — ram 3

print(f"AOI (Skåne): {AOI}")
print(f"År: {YEAR} (4 temporala ramar)")

## 3. Kör multitemporal analys

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/multitemporal_lulc",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=REFERENCE_DATE,
    coords=AOI,
    multitemporal=True,
    num_temporal_frames=4,
    lpis_year=YEAR,
)

if "prithvi" in result.analyses:
    prithvi = result.analyses["prithvi"].data
    seg = prithvi.get("segmentation")
    if seg is not None:
        print(f"LULC-kartform: {seg.shape}")
        unique_classes = set(seg.flatten().tolist())
        print(f"Klasser närvarande: {sorted(unique_classes)}")

## 4. Visualisering

In [ ]:
# 23-klassers färgpalett
CLASS_NAMES = [
    "bakgrund", "tallskog", "granskog", "lövskog", "blandskog", "sumpskog",
    "tillfälligt ej skog", "våtmark", "öppen mark", "bebyggelse", "vatten",
    "vete", "korn", "havre", "oljeväxter", "slåttervall", "bete",
    "potatis", "sockerbetor", "trindsäd", "råg", "majs", "hygge"
]

# Färgpalett inspirerad av NMD + agri-standard
CLASS_COLORS = [
    "#000000",  # bakgrund
    "#004d00", "#006b00", "#8fbc8f", "#4f7f4f", "#305030",  # skog
    "#ffffbb",  # tillf. ej skog
    "#4d9fff",  # våtmark
    "#ddeeb6",  # öppen mark
    "#aa0000",  # bebyggelse
    "#0066cc",  # vatten
    "#ffd700", "#daa520", "#b8860b", "#ff6347", "#90ee90", "#228b22",  # grödor
    "#dda0dd", "#ee82ee", "#ba55d3", "#8b4513", "#ffa500",
    "#8b0000",  # hygge
]

cmap = mcolors.ListedColormap(CLASS_COLORS[:len(CLASS_NAMES)])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(result.rgb)
axes[0].set_title(f"RGB ({REFERENCE_DATE})")
axes[0].axis("off")

if "prithvi" in result.analyses:
    seg = result.analyses["prithvi"].data.get("segmentation")
    if seg is not None:
        axes[1].imshow(seg, cmap=cmap, vmin=0, vmax=len(CLASS_NAMES)-1)
        axes[1].set_title("Prithvi-EO 23-klass LULC")
axes[1].axis("off")

if "nmd" in result.analyses:
    nmd_overlay = result.analyses["nmd"].data.get("overlay")
    if nmd_overlay is not None:
        axes[2].imshow(nmd_overlay)
        axes[2].set_title("NMD ground truth")
axes[2].axis("off")

plt.tight_layout()
plt.show()

# Legend
fig_leg, ax_leg = plt.subplots(figsize=(14, 2))
patches = [plt.Rectangle((0, 0), 1, 1, facecolor=CLASS_COLORS[i]) for i in range(len(CLASS_NAMES))]
ax_leg.legend(patches, CLASS_NAMES, loc="center", ncol=6, fontsize=8, frameon=False)
ax_leg.axis("off")
plt.tight_layout()
plt.show()

## 5. Tolkning

**Varför multitemporal + foundation model?**

| Ansats | mIoU (skåne-testset) | Kommentar |
|--------|---------------------|-----------|
| U-Net, 10-klass, single-frame | 44.14% | Legacy baseline |
| Prithvi single-frame, 19-klass | 37.5% | Fler klasser → svårare |
| Prithvi 4-frame multitemporal, 23-klass | **TBD (pågående)** | Förväntas överträffa |

**Dual-head arkitektur:**
1. **Huvud 1 — LULC:** 23-klass softmax (focal loss)
2. **Huvud 2 — Avverkningsmogen:** binär sannolikhet per skogspixel (sigmoid, BCE loss)

Modellen lär sig att känna igen *spektral/aux-signatur för avverkningsmogen skog* — underlag till Skogsstyrelsens planering.

**Samhällsnytta:**
- **Jordbruksverket:** 100% automatiserad LPIS-bekräftelse
- **Naturvårdsverket:** årlig NMD-uppdatering (idag 4-års-cykel)
- **Skogsstyrelsen:** underlag för avverkningsstrategier
- **Forskning:** publik multitemporal benchmark för svenska förhållanden

**Nästa steg:**
- Skala till hela Sverige → nationell LULC-karta
- Publicera som årlig leverans via DES
- Integrera SAR-data (Sentinel-1) för moln-tolerant inferens
- Föra över lärdomar till Copernicus CLMS-projekt

### Länkar
- [ImintEngine training pipeline](https://github.com/TobiasEdman/imintengine/tree/main/imint/training)
- [Prithvi-EO 2.0 paper (IBM/NASA)](https://arxiv.org/abs/2310.18660)
- [NMD hos Naturvårdsverket](https://www.naturvardsverket.se/nmd)
- [LPIS hos Jordbruksverket](https://jordbruksverket.se)